# MedSense — Phase 1: EDA on Symptom2Disease

Goal: understand class balance, text length, vocabulary before fine-tuning.

Drop the Kaggle `Symptom2Disease.csv` into `./data/` before running this notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "./data/Symptom2Disease.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Column check & cleanup

The raw Kaggle file typically has an unnamed index column plus `label` and `text`. Adjust the rename below if your column names differ.

In [ ]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df.columns = [c.strip().lower() for c in df.columns]
print(df.columns.tolist())
df = df.dropna(subset=['text', 'label']).drop_duplicates(subset=['text'])
print('Rows after cleanup:', len(df))

## Class balance

In [ ]:
counts = df['label'].value_counts()
print(f"{counts.shape[0]} distinct conditions")
print(counts)

plt.figure(figsize=(10, 6))
counts.plot(kind='barh')
plt.title('Samples per condition')
plt.xlabel('count')
plt.tight_layout()
plt.show()

## Text length distribution

Informs `max_length` for tokenization in `train.py`.

In [ ]:
df['word_count'] = df['text'].str.split().apply(len)
print(df['word_count'].describe())

plt.figure(figsize=(8, 5))
df['word_count'].hist(bins=30)
plt.title('Symptom text length (words)')
plt.xlabel('word count')
plt.ylabel('frequency')
plt.show()

## Most common keywords (rough pass)

Simple frequency count for now — the analytics dashboard endpoint `/api/analytics/symptom-keywords` does a similar aggregation later, on live logged queries.

In [ ]:
from collections import Counter
import re

STOPWORDS = {'the','a','an','and','or','of','to','in','for','on','with','is','are','my','i','me','have','has','been','it','this','that'}

words = Counter()
for text in df['text']:
    tokens = re.findall(r'[a-z]+', text.lower())
    words.update(t for t in tokens if t not in STOPWORDS and len(t) > 2)

for word, freq in words.most_common(25):
    print(f'{word:20s} {freq}')

## Save cleaned dataset

`train.py` consumes this directly.

In [ ]:
df[['text', 'label']].to_csv('./data/Symptom2Disease_clean.csv', index=False)
print('Saved cleaned dataset.')

## Notes / risks observed

- If any condition has very few samples (<10), consider merging rare classes or flagging for augmentation before stratified splitting in `train.py`.
- Record final class count and imbalance ratio here after running against the real file, for the README's dataset section.